# 🔤 Tokenizers Atención

Cuánto "pesa" una palabra para un LLM? Es lo mismo "gato" que "gatos"? Las computadoras **nunca ven texto**, ven números

## 0. Imports y configuración

In [1]:
import numpy as np

import torch
from transformers import AutoModel, AutoTokenizer

from bertviz import head_view, model_view
from helpers.viz import plot_token_comparison, plot_transformer_attention

## 1. TOKEN

Antes de cargar un modelo, necesitamos entender **cómo convierte texto en números**.

Un **tokenizer** divide el texto en unidades llamadas *tokens*. Estos tokens no son necesariamente palabras: pueden ser sílabas, partes de palabras, o incluso caracteres individuales.

El proceso completo es:

```
Texto  →  [Tokenizer]  →  IDs numéricos  →  [Embedding layer]  →  Vectores  →  [Transformer]
```

> **¿Por qué no usar palabras completas?** Un vocabulario de todas las palabras en todos los idiomas sería enorme e impráctico. Los tokenizers modernos usan algoritmos como **BPE** (Byte-Pair Encoding) o **SentencePiece** para encontrar sub-palabras frecuentes.


## 2. Gemma 2 Tokenizer

Usaremos el tokenizer oficial de Google para Gemma 2. Este es el mismo tokenizer que usa Gemma 4 internamente.

> **Nota**: Requiere haber aceptado los términos en [huggingface.co/google/gemma-2-2b](https://huggingface.co/google/gemma-2-2b) y haber corrido `hf auth login`.


In [2]:
MODEL_ID = "google/gemma-2-2b"

print(f"Cargando tokenizer de {MODEL_ID}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

print(f"✅ Tokenizer listo")
print(f"   Vocabulario: {tokenizer.vocab_size:,} tokens")
print(f"   Algoritmo  : {type(tokenizer).__name__}")
print(f"   Token BOS  : {tokenizer.bos_token!r} (ID: {tokenizer.bos_token_id})")
print(f"   Token EOS  : {tokenizer.eos_token!r} (ID: {tokenizer.eos_token_id})")

Cargando tokenizer de google/gemma-2-2b...
✅ Tokenizer listo
   Vocabulario: 256,000 tokens
   Algoritmo  : GemmaTokenizer
   Token BOS  : '<bos>' (ID: 2)
   Token EOS  : '<eos>' (ID: 1)


## 3. Primera vez tokenizando?

In [3]:
FRASE = "Me atrapaste, en efecto es cine... pero no dije buen cine"

encoded = tokenizer(FRASE, return_tensors=None)

encoded

{'input_ids': [2, 1898, 128014, 4555, 235269, 659, 41064, 875, 37797, 955, 6703, 793, 81593, 23116, 37797], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [4]:
ids = encoded["input_ids"]
ids

[2,
 1898,
 128014,
 4555,
 235269,
 659,
 41064,
 875,
 37797,
 955,
 6703,
 793,
 81593,
 23116,
 37797]

In [5]:
tokens = tokenizer.convert_ids_to_tokens(ids)

print(f"Texto original : {FRASE!r}")
print(f"Número de IDs  : {len(ids)}")
print()
print(f"{'#':<5} {'Token':<20} {'ID'}")
print("-" * 35)
for i, (tok, id_) in enumerate(zip(tokens, ids)):
    print(f"{i:<5} {tok!r:<20} {id_}")

Texto original : 'Me atrapaste, en efecto es cine... pero no dije buen cine'
Número de IDs  : 15

#     Token                ID
-----------------------------------
0     '<bos>'              2
1     'Me'                 1898
2     '▁atrap'             128014
3     'aste'               4555
4     ','                  235269
5     '▁en'                659
6     '▁efecto'            41064
7     '▁es'                875
8     '▁cine'              37797
9     '...'                955
10    '▁pero'              6703
11    '▁no'                793
12    '▁dije'              81593
13    '▁buen'              23116
14    '▁cine'              37797


## 5. Pero, importa el idioma?

Los LLMs modernos son multilingues, pero no todos los idiomas "cuestan" lo mismo en tokens.


In [6]:
textos_para_comparar = [
    "Me atrapaste, en efecto es cine... pero no dije buen cine",  # Español (Original)
    "You caught me, indeed it is cinema... but I didn't say good cinema",  # Inglés
    "Du hast mich erwischt, in der Tat, es ist Kino... aber ich habe nicht gutes Kino gesagt",  # Alemán
    "Tu m'as eu, en effet c'est du cinéma... mais je n'ai pas dit du bon cinéma",  # Francés
    "捕まったよ、確かにこれは映画だ... でも良い映画だとは言ってない。"  # Japonés
]

print("Texto → Número de tokens (Gemma 2):")
print("-" * 65)
for texto in textos_para_comparar:
    ids = tokenizer(texto)["input_ids"]
    print(f"  [{len(ids):>3} tokens] {texto}")


Texto → Número de tokens (Gemma 2):
-----------------------------------------------------------------
  [ 15 tokens] Me atrapaste, en efecto es cine... pero no dije buen cine
  [ 18 tokens] You caught me, indeed it is cinema... but I didn't say good cinema
  [ 23 tokens] Du hast mich erwischt, in der Tat, es ist Kino... aber ich habe nicht gutes Kino gesagt
  [ 25 tokens] Tu m'as eu, en effet c'est du cinéma... mais je n'ai pas dit du bon cinéma
  [ 18 tokens] 捕まったよ、確かにこれは映画だ... でも良い映画だとは言ってない。


## 6. Visualización comparativa: Gemma 2 vs GPT-2

Comparemos dos tokenizers de arquitecturas diferentes para el mismo texto.


In [7]:
tokenizer_gpt2 = AutoTokenizer.from_pretrained("gpt2")  # modelo público, sin login

textos = [
    "Machine learning",
    "Aprendizaje automático",
    "def suma(a, b): return a + b",
    "¿Cuántos planetas hay en el sistema solar?",
    FRASE
]

fig = plot_token_comparison(
    textos,
    tokenizers={
        "Gemma 2 (SentencePiece)": tokenizer,
        "GPT-2 (BPE)": tokenizer_gpt2,
    },
    title="Gemma 2 vs GPT-2: ¿Quién es más eficiente?",
)
fig.show()

## 7. Del token al texto: decodificación

El proceso inverso: tomar IDs numéricos y reconstruir el texto original.


In [8]:
ids_ejemplo = [987, 893, 51528, 822, 37797, 235269, 180087, 235265]

# Decodificación
texto_reconstruido = tokenizer.decode(ids_ejemplo, skip_special_tokens=True)
tokens_raw = tokenizer.convert_ids_to_tokens(ids_ejemplo)

print(f"IDs de entrada: {ids_ejemplo}")
print(f"Tokens crudos : {tokens_raw}")
print(f"Texto final   : {repr(texto_reconstruido)}")
print()
print("✅ El tokenizer es completamente reversible — sin pérdida de información.")

IDs de entrada: [987, 893, 51528, 822, 37797, 235269, 180087, 235265]
Tokens crudos : ['Re', 'vi', 'vió', '▁el', '▁cine', ',', '▁señores', '.']
Texto final   : 'Revivió el cine, señores.'

✅ El tokenizer es completamente reversible — sin pérdida de información.


## 8. **Attention is all you need**

Hasta ahora vimos **cómo se codifica** el texto. Ahora el paso siguiente: dentro del Transformer, cada token "mira" a los demás para entender el contexto.

> **Intuición**: La palabra "banco" en "Me senté en el banco del parque" vs "Fui al banco a sacar dinero" tiene significados distintos. El mecanismo de atención permite al modelo distinguirlos mirando los tokens vecinos.

### ¿Qué es la atención?

Cada token produce tres vectores (como perfiles de citas rápidas):
* **Query (Q)**: "¿Qué contexto estoy buscando?"
* **Key (K)**: "¿Qué contexto ofrezco?"
* **Value (V)**: "La información real que comparto."

La atención se calcula con esta famosa fórmula:

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

**Traduciendo la fórmula al español:**
1. **$QK^T$ (El Match):** Multiplicamos lo que busco ($Q$) por lo que otros ofrecen ($K$). La $T$ es solo la Transpuesta para que las piezas encajen.
2. **$\sqrt{d_k}$ (El Termostato):** Dividimos entre la raíz cuadrada del tamaño del vector. Esto "enfría" los números grandes para que la IA no tome decisiones demasiado extremas.
3. **$\text{softmax}$ (El Porcentaje):** Convierte el resultado en porcentajes que suman 100%. Así el modelo dice: "Le pondré 80% de atención a 'parque' y 20% a 'senté'".
4. **$V$ (El Resumen):** Multiplicamos esos porcentajes por la información real de los tokens ($V$).

Para visualizarlo, necesitamos cargar el modelo completo con `output_attentions=True`. Por el tamaño de Gemma 2, usaremos **DistilBERT** como demostración visual — el mecanismo subyacente es idéntico.

In [9]:
# DistilBERT: modelo compacto, mismo mecanismo de atención que Gemma
BERT_MODEL = "distilbert-base-uncased"

print(f"Cargando {BERT_MODEL}...")
bert_tokenizer = AutoTokenizer.from_pretrained(BERT_MODEL)
bert_model = AutoModel.from_pretrained(BERT_MODEL, output_attentions=True)
bert_model.eval()
print("✅ Modelo listo")

Cargando distilbert-base-uncased...


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.weight  | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Modelo listo


## 9. Visualizar la atención con BertViz

`bertviz` es la herramienta estándar de la industria para inspeccionar la atención interna de Transformers.


In [10]:
FRASE_ATENCION = "The animal didn't cross the street because it was too tired!"

# La función se encarga de:
# 1. Tokenizar e inferir (con torch.no_grad y output_attentions=True)
# 2. Buscar la "magia": la capa y cabeza donde la conexión sea más fuerte
# 3. Generar el mapa de calor interactivo
fig, atenciones, tokens = plot_transformer_attention(
    model=bert_model,
    tokenizer=bert_tokenizer,
    text="The animal didn't cross the street because it was too tired!",
    find_connection=["it", "animal"]
)

fig.show()

🕵️‍♂️ ¡Magia encontrada! Para 'it' -> 'animal':
   Usa la Capa 4, Cabeza 1 (Atención: 63.8%)


`bertviz` ofrece una visualización interactiva más rica, donde puedes cambiar de capa y cabeza en tiempo real.

In [11]:
head_view(atenciones, tokens)

<IPython.core.display.Javascript object>

## 10. ¿Qué aprendimos hoy?

| Concepto | Resumen |
|----------|---------|
| **Token** | Unidad mínima de texto que procesa un LLM. No es necesariamente una palabra. |
| **Tokenizer (SentencePiece/BPE)** | Algoritmo que divide texto en tokens balanceando vocabulario y eficiencia. |
| **Vocabulario** | Gemma 2 tiene ~256K tokens. GPT-2 tiene ~50K. |
| **Idioma y tokens** | Los idiomas con más datos de entrenamiento necesitan menos tokens por oración. |
| **Tokens especiales** | BOS, EOS, PAD → controlan el flujo del modelo, no son "contenido". |
| **Mecanismo de atención** | Cada token "mira" a todos los demás para entender el contexto. Múltiples cabezas = múltiples perspectivas simultáneas. |